In [1]:
# %% [markdown]
# # Machine Learning for Ischemic Stroke Risk Prediction
#
# **Authors:** Suleman Zakaria, Georgina Buabeng Boateng, Dontoh Evans, Michael Adu Yeboah
# **Date:** January 2026
#
# ## Abstract
# This notebook implements the methodology described in the research article. It develops and evaluates machine learning models (Logistic Regression, Random Forest, Gradient Boosting) for ischemic stroke risk prediction. It specifically addresses:
# 1.  Missing BMI values (Mean Imputation).
# 2.  Categorical Encoding (Binary, Ordinal, Label).
# 3.  Class Imbalance (SMOTE).
# 4.  Evaluation of performance before and after SMOTE.

# %% [markdown]
# ## 1. Setup and Library Imports
# Importing necessary libraries for data manipulation, visualization, and machine learning.

# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn for preprocessing and modeling
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Imbalanced Learning (SMOTE)
import importlib, sys, subprocess
try:
    _module = importlib.import_module('imblearn.over_sampling')
    SMOTE = getattr(_module, 'SMOTE')
except Exception:
    print("imbalanced-learn not found; installing imbalanced-learn...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "imbalanced-learn"])
    _module = importlib.import_module('imblearn.over_sampling')
    SMOTE = getattr(_module, 'SMOTE')

import warnings
warnings.filterwarnings('ignore')

# %% [markdown]
# ## 2. Methodology: Data Loading and Extraction
# **Section 2.1 & 2.2:** Loading the Kaggle Stroke Prediction Dataset.
# * **Source:** healthcare-dataset-stroke-data.csv
# * **Cleaning:** Dropping the 'id' column as it has no predictive value.

# %%
# Load the dataset
# Note: Ensure the csv file is in the same directory
df = pd.read_csv('healthcare-dataset-stroke-data.csv')

# Drop ID column (irrelevant identifier)
if 'id' in df.columns:
    df = df.drop(columns=['id'])

print("Dataset Shape:", df.shape)
df.head()

# %% [markdown]
# ## 3. Data Preprocessing
# **Section 2.4:** Addressing missing values, categorical encoding, and scaling.

# %%
# --- 3.1 Handling Missing Values (BMI) ---
# "Data preprocessing addressed missing BMI values through mean imputation"
imputer = SimpleImputer(strategy='mean')
df['bmi'] = imputer.fit_transform(df[['bmi']])

# --- 3.2 Categorical Encoding ---
# "Categorical variables through appropriate encoding (e.g., gender, work_type, smoking_status)"

# A. Binary Encoding (Gender, Ever_Married, Residence_Type)
# Mapping for binary consistency
df['gender'] = df['gender'].map({'Male': 0, 'Female': 1, 'Other': 2})
df['ever_married'] = df['ever_married'].map({'No': 0, 'Yes': 1})
df['residence_type'] = df['residence_type'].map({'Rural': 0, 'Urban': 1})

# Handle 'Other' gender if necessary (usually very few, can be dropped or kept)
df = df[df['gender'] != 2] # Removing 'Other' for cleaner binary classification if preferred

# B. Label Encoding for Work Type
# "Label encoding was utilized for work_type"
le = LabelEncoder()
df['work_type'] = le.fit_transform(df['work_type'])

# C. Ordinal Encoding for Smoking Status
# "Ordinal encoding was implemented for smoking_status to reflect potential risk level"
# Mapping: Unknown=0, never smoked=1, formerly smoked=2, smokes=3
smoking_map = {'Unknown': 0, 'never smoked': 1, 'formerly smoked': 2, 'smokes': 3}
df['smoking_status'] = df['smoking_status'].map(smoking_map)

# --- 3.3 Feature Scaling ---
# "Min-Max normalization was applied to the numerical features"
scaler = MinMaxScaler()
numerical_cols = ['age', 'avg_glucose_level', 'bmi']
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

# Verify Preprocessing
print("Data after preprocessing:")
display(df.head())

# %% [markdown]
# ## 4. Model Training Phase 1: Before Addressing Class Imbalance
# **Section 2.5 & Results (Table 2):** Training models on the raw, imbalanced data.
#
# * **Observation:** We expect Logistic Regression and Random Forest to struggle with the minority class (Stroke=1).

# %%
# Split Data
X = df.drop(columns=['stroke'])
y = df['stroke']

# Stratified Split to maintain class ratio in test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Define Models
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

# Helper function to evaluate models
def evaluate_models(models, X_tr, y_tr, X_te, y_te, phase_name):
    results = []
    for name, model in models.items():
        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)
        probs = model.predict_proba(X_te)[:, 1]

        results.append({
            "Model": name,
            "Accuracy": accuracy_score(y_te, preds),
            "Precision": precision_score(y_te, preds, zero_division=0),
            "Recall": recall_score(y_te, preds),
            "F1-score": f1_score(y_te, preds),
            "ROC AUC": roc_auc_score(y_te, probs)
        })

    results_df = pd.DataFrame(results)
    print(f"\n--- {phase_name} Results ---")
    display(results_df)
    return results_df

# Run Evaluation Phase 1
results_phase_1 = evaluate_models(models, X_train, y_train, X_test, y_test, "Phase 1 (Before SMOTE)")

# %% [markdown]
# ## 5. Handling Imbalance (SMOTE)
# **Section 2.4:** Applying Synthetic Minority Oversampling Technique (SMOTE).
# * **Goal:** To mitigate bias towards the majority class.
# * **Note:** SMOTE is applied **only to the training data** to prevent data leakage.

# %%
print(f"Original Training Class Distribution: \n{y_train.value_counts()}")

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Balanced Training Class Distribution (After SMOTE): \n{y_train_smote.value_counts()}")

# %% [markdown]
# ## 6. Model Training Phase 2: After Addressing Class Imbalance
# **Results (Table 3):** Models are retrained on the SMOTE-balanced data.
# * **Expectation:** Significant improvement in **Recall** and **F1-score**.

# %%
# Retrain models on SMOTE data
results_phase_2 = evaluate_models(models, X_train_smote, y_train_smote, X_test, y_test, "Phase 2 (After SMOTE)")

# %% [markdown]
# ## 7. Model Training Phase 3: Feature Interactions
# **Results (Table 4):** Testing if interaction terms improve the best model (Logistic Regression).
# * "Interaction terms were added to the features... no performance improvement was yielded."

# %%
from sklearn.preprocessing import PolynomialFeatures

# Create Interaction Terms (degree=2, interaction_only=True)
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_train_inter = poly.fit_transform(X_train_smote)
X_test_inter = poly.transform(X_test)

# Train Logistic Regression with Interactions
lr_inter = LogisticRegression(random_state=42, max_iter=2000)
lr_inter.fit(X_train_inter, y_train_smote)

# Evaluate
preds = lr_inter.predict(X_test_inter)
probs = lr_inter.predict_proba(X_test_inter)[:, 1]

# Create comparison dataframe manually to match Table 4
phase_3_data = {
    "Model": ["Logistic Regression (SMOTE + Interactions)"],
    "Accuracy": [accuracy_score(y_test, preds)],
    "Precision": [precision_score(y_test, preds)],
    "Recall": [recall_score(y_test, preds)],
    "F1-score": [f1_score(y_test, preds)],
    "ROC AUC": [roc_auc_score(y_test, probs)]
}

results_phase_3 = pd.DataFrame(phase_3_data)
print("\n--- Phase 3 Results (Interactions) ---")
display(results_phase_3)

# %% [markdown]
# ## 8. Visualization: ROC Curve Comparison
# **Figure 2:** Comparison of ROC AUC Scores Before and After Applying SMOTE.

# %%
plt.figure(figsize=(10, 6))

# Plotting ROC for Logistic Regression (Before SMOTE)
model_orig = models["Logistic Regression"]
model_orig.fit(X_train, y_train)
fpr1, tpr1, _ = roc_curve(y_test, model_orig.predict_proba(X_test)[:, 1])
plt.plot(fpr1, tpr1, label=f'LogReg (Before SMOTE), AUC={roc_auc_score(y_test, model_orig.predict_proba(X_test)[:, 1]):.2f}', linestyle='--')

# Plotting ROC for Logistic Regression (After SMOTE)
model_smote = models["Logistic Regression"] # This object was trained on SMOTE data in Phase 2 loop
model_smote.fit(X_train_smote, y_train_smote)
fpr2, tpr2, _ = roc_curve(y_test, model_smote.predict_proba(X_test)[:, 1])
plt.plot(fpr2, tpr2, label=f'LogReg (After SMOTE), AUC={roc_auc_score(y_test, model_smote.predict_proba(X_test)[:, 1]):.2f}', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison: Impact of SMOTE')
plt.legend()
plt.grid(True)
plt.show()

# %% [markdown]
# ## 9. Conclusion
#
# Based on the results above:
# 1.  **Before SMOTE:** Models achieved high accuracy (~94%) but failed to detect strokes (Recall ~0).
# 2.  **After SMOTE:** Logistic Regression achieved the highest Recall (~0.71), making it the preferred model for screening.
# 3.  **Interactions:** Added no significant value to the metrics.
#
# **Recommendation:** The SMOTE-balanced Logistic Regression model is selected for deployment.

ModuleNotFoundError: No module named 'imblearn'